# 05. Sampling topology

**What you will achieve:** Explore how domains are partitioned and how fixed, deterministic or random topology changes the grouping of stochastic perturbations.

**For your own data:** replace `INPUT_FILE` first; then change the brief settings cell as needed. This notebook uses `tests/fixtures/input_2.leapr` and the real TARA Core interfaces. There is no backend execution here.


## Scientific idea: domains and topology

TARA partitions the energy axis into domains, often around natural spectral features. The topology strategy specifies **whether and how those boundaries vary across the ensemble**: `fixed` retains the original partition, `deterministic` selects reproducible variations, and `random` varies them stochastically.

This example chooses `deterministic`, so the source of the partition changes can be inspected in `ensemble.sampling_blocks`. A block may cover multiple samples sharing one topology; this does not mean those samples have identical density curves. Distinguish *domain selection* from *amplitude/energy multipliers*.


## 1. Locate TARA

The next cell finds TARA and imports the sampling APIs.

In [ ]:
# Locate the TARA repository so this notebook works from any of its subdirectories.
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if
     (p / 'core' / 'config.py').is_file() and
     (p / 'core' / 'sampler' / 'concrete_samplers.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter inside your TARA repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import TaraConfig
from core.data.input_pdos import PhononSpectra
from core.sampler.concrete_samplers import TaraSampler
from input_output.factory import load_file_object

print(f'TARA root: {PROJECT_ROOT}')


## 2. Select the required inputs

**Required settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `INPUT_FILE` | `tests/fixtures/input_2.leapr` | Source of the reference phonon spectrum. |

In [ ]:
INPUT_FILE = PROJECT_ROOT / 'tests' / 'fixtures' / 'input_2.leapr'
if not INPUT_FILE.is_file():
    raise FileNotFoundError(f'Example LEAPR fixture is missing: {INPUT_FILE}')

## 3. Load the graphite reference

Load the selected input and prepare the normalized reference PDOS.

In [ ]:
file_template, raw_pdos = load_file_object(filepath=INPUT_FILE)
reference_spectrum = PhononSpectra(pdos=raw_pdos, auto_normalize=True)
print(f'Input: {INPUT_FILE.name}')

## 4. Set up this experiment

Change the domain splitting strategy to study how energy regions share perturbations.

**Required settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `MAX_ENERGY_SHIFT_MEV` | `1.0` | Maximum phonon-energy displacement in meV. |
| `MAX_AMPLITUDE_SHIFT_PCT` | `5.0` | Maximum relative amplitude change in percent. |
| `N_SAMPLES` | `3` | Number of stochastic samples; the reference is stored separately. |

In [ ]:
MAX_ENERGY_SHIFT_MEV = 1.0
MAX_AMPLITUDE_SHIFT_PCT = 5.0
N_SAMPLES = 3

**Optional settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `RANDOM_SEED` | `0` | Reproducible seed passed to TaraConfig for Latin hypercube sampling. |
| `BATCH_SIZE` | `512` | Number of samples processed per internal chunk. |
| `AMPLITUDE_METHOD` | `'fast'` | Amplitude perturbation method. |
| `SPATIAL_METHOD` | `'diffeomorphic'` | Energy-grid perturbation method. |
| `DOMAIN_SPLITTING_STRATEGY` | `'deterministic'` | Rule that varies the spectral partition across sub-ensembles. |

In [ ]:
RANDOM_SEED = 0
BATCH_SIZE = 512
AMPLITUDE_METHOD = 'fast'
SPATIAL_METHOD = 'diffeomorphic'
DOMAIN_SPLITTING_STRATEGY = 'deterministic'

In [ ]:
# Validate all perturbation bounds and sampling strategies using TaraConfig.
configuration = TaraConfig(
    max_energy_shift_mev=MAX_ENERGY_SHIFT_MEV,
    max_amplitude_shift_pct=MAX_AMPLITUDE_SHIFT_PCT,
    n_samples=N_SAMPLES,
    random_seed=RANDOM_SEED,
)

# Initialize TaraSampler with the selected amplitude and spatial methods.
sampler = TaraSampler(
    config=configuration,
    spectra=reference_spectrum,
    amplitude_method=AMPLITUDE_METHOD,
    spatial_method=SPATIAL_METHOD,
)

## 5. Generate the ensemble

The actual sampling uses a reproducible seed. `batch_size` controls numerical chunking and is not a limit on sample count. The baseline remains in `ensemble.reference`, and every `ensemble[i]` is a stochastic realization.

In [ ]:
from notebooks.tara_plotting import plot_PDOS
import matplotlib.pyplot as plt

In [ ]:
# Generate the stochastic spectra and preserve the original baseline separately.
ensemble = sampler.generate_ensemble(batch_size=BATCH_SIZE)
assert len(ensemble) == N_SAMPLES
assert ensemble.reference is not None
print(f'Stochastic realizations: {len(ensemble)}')
print('The reference is stored at ensemble.reference, never at ensemble[0].')

In [ ]:
# ---------------------------------------------------------
# Step 5: Plot PDOS and the Different Partition Topologies
# ---------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 6))

# Reference PDOS and all generated samples.
plot_PDOS(
    ax,
    list(ensemble),
    ensemble.reference,
    title="Sampling Topology: Reference vs Samples",
)

# Each sampling block can have a different domain partition.
blocks = ensemble.sampling_blocks
colors = plt.cm.tab10.colors

# Reserve the lower part of the plot for topology bands.
band_height = 0.99 / len(blocks)

for block_index, block in enumerate(blocks):
    bottom = block_index * band_height
    top = bottom + band_height * 0.85

    for domain_index, (start, end) in enumerate(block.boundaries):
        ax.axvspan(
            start,
            end,
            ymin=bottom,
            ymax=top,
            color=colors[block_index % len(colors)],
            alpha=0.35 if domain_index % 2 == 0 else 0.65,
            edgecolor="black",
            linewidth=0.6,
            label=(
                f"Topology {block_index + 1} "
                f"(samples {block.start_index + 1}–{block.end_index})"
                if domain_index == 0
                else "_nolegend_"
            ),
        )

ax.legend(loc="best")
plt.tight_layout()
plt.show()

## 6. Zoom into a Region with Different Partition Topologies

The domain splitting strategy determines how the energy axis is partitioned for the generated samples. Consequently, different samples may use different domain boundaries.

We focus on the energy interval **0.067–0.124 eV**, where these differences can be examined in detail.

The figure compares the reference PDOS with all stochastic realizations in the same energy window. Dashed vertical lines identify the partition boundaries recorded in each sampling block, allowing us to observe how different domain topologies influence the perturbations of nearby spectral peaks.

**Optional display settings**

| Setting | Example value | What it controls |
| --- | --- | --- |
| `ZOOM_RANGE_EV` | `(0.067, 0.124)` | Visible energy interval in eV. |

In [ ]:
# ---------------------------------------------------------
# Step 6: Zoom into a Region with Different Topologies
# ---------------------------------------------------------

ZOOM_RANGE_EV = (0.067, 0.124)

fig, ax = plt.subplots(figsize=(12, 6))

plot_PDOS(
    ax,
    list(ensemble),
    ensemble.reference,
    title="Sampling Topology — Local PDOS Comparison",
)

# Display the boundaries associated with each sampling block.
colors = plt.cm.tab10.colors

for block_index, block in enumerate(ensemble.sampling_blocks):
    boundaries = np.asarray(block.boundaries)

    # Unique domain edges within the selected energy window.
    edges = np.unique(boundaries.ravel())
    visible_edges = edges[
        (edges > ZOOM_RANGE_EV[0])
        & (edges < ZOOM_RANGE_EV[1])
    ]

    for edge_index, edge in enumerate(visible_edges):
        ax.axvline(
            edge,
            color=colors[block_index % len(colors)],
            linestyle="--",
            linewidth=1.3,
            alpha=0.7,
            label=(
                f"Topology — samples "
                f"{block.start_index + 1}–{block.end_index}"
                if edge_index == 0
                else "_nolegend_"
            ),
        )

ax.set_xlim(*ZOOM_RANGE_EV)
ax.legend(loc="best")

plt.tight_layout()
plt.show()